# fin-anim-agent — whiteboard icon + hand generator (free Colab GPU)

Generates whiteboard-marker-style doodle PNGs for `whiteboard_explainer`'s `"image"`
visual mode, **and** a realistic hand-and-pencil PNG to replace the default stylized
vector hand — both with a local Stable Diffusion model on Colab's free T4 GPU, no paid
API/MCP credits.

**v2 changes** (after the first batch had two real problems): real background removal
via `rembg` (a segmentation model, not a naive white-color threshold — the first batch's
backgrounds were paper-textured/hatched, not flat white, so thresholding barely worked),
and multiple candidates per icon with a contact-sheet preview so you pick the best one
instead of keeping whatever the first random seed produced.

**v3 changes:** added the realistic hand section near the end.

**Before running:** `Runtime` menu -> `Change runtime type` -> Hardware accelerator = **T4 GPU** (or better).

**Recommendation:** skip regenerating `dollar`, `arrow_up`, `arrow_down` here — comment
them out of `ICON_CONCEPTS` below. Stable Diffusion 1.5 (a general-purpose model, not a
symbol/glyph specialist) reliably mangles simple glyphs and single-stroke arrows; the
existing vector icons for these three (`_icon_dollar`, `_icon_arrow_up`, `_icon_arrow_down`
in `whiteboard_assets.py`) are already exact and free, with no generation-failure risk.
Spend Colab time on concepts that actually benefit from more detail than simple
primitives give: `piggy_bank`, `scale`, and anything you add beyond the original 11.

**After running:** download `whiteboard_icons.zip` (last cell), unzip it, and copy the PNGs
into this repo's `assets/whiteboard_icons/` folder. Then reference an icon in a beat:

```json
{"visual": "image", "image_path": "assets/whiteboard_icons/piggy_bank.png", "label": "Save $100", ...}
```

`hand.png`, if present at `assets/whiteboard_icons/hand.png`, is picked up automatically
by every beat's render — no per-beat reference needed. For a richer multi-element scene
in one beat, see `"svg"` visual mode (SKILL.md Step 1b) with a free CC0 illustration from
[undraw.co](https://undraw.co) — that's a separate, non-Colab step (unDraw ships ready SVGs).

In [ ]:
!pip install -q diffusers transformers accelerate safetensors rembg onnxruntime

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU, then re-run"

MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, safety_checker=None)
pipe = pipe.to("cuda")
print("Model loaded on", pipe.device)

## Part 1 — icons

In [ ]:
# Concept per icon name — keep these keys in sync with WHITEBOARD_ICONS in
# skills/fin-anim/scripts/schema.py. Each concept is written literally and
# narrowly ("a classic X, side view, isolated pictogram") rather than loosely,
# since vague phrasing is what produced the wrong-concept results in v1
# (arrow_up as a fur texture, scale as hanging pouches, etc.) — precise nouns
# and an explicit single-subject framing steer the model much more reliably.
#
# Comment out dollar/arrow_up/arrow_down — see the markdown cell above for why.
ICON_CONCEPTS = {
    # "dollar": "a single dollar currency symbol character, bold and simple",
    # "arrow_up": "a single straight vertical arrow pointing straight up, like a keyboard up-arrow key icon",
    # "arrow_down": "a single straight vertical arrow pointing straight down, like a keyboard down-arrow key icon",
    "clock": "a simple round analog clock face with an hour hand and a minute hand, side view",
    "house": "a simple single-story house with a triangular roof and one door, front view, no other objects",
    "piggy_bank": "a classic piggy bank money box shaped like a round pig, with a coin slot on top and four short legs, side view",
    "coin_stack": "a stack of three round coins, side view",
    "calculator": "a pocket calculator with a rectangular screen and a grid of square buttons, front view",
    "chart_bar": "a simple bar chart with four vertical bars of different heights on a baseline, no text",
    "lightbulb": "a single incandescent lightbulb with a screw base and a few short rays of light around it",
    "scale": "a classic balance scale with a center pole, a horizontal beam, and two hanging pans on chains at each end",
}

PROMPT_TEMPLATE = (
    "{concept}, isolated pictogram, single subject centered in frame, "
    "simple black ink line drawing, whiteboard marker sketch style, "
    "thick clean bold outlines, minimalist icon, flat solid white background, "
    "no shading, no color, no gradient, no texture"
)
NEGATIVE_PROMPT = (
    "photo, photorealistic, 3d render, color, shading, gradient, text, watermark, "
    "signature, blurry, realistic, textured background, paper texture, canvas texture, "
    "grain, hatching, cross-hatching, patterned background, spiral pattern, gibberish text, "
    "distorted, deformed, extra limbs, fur, hair, multiple objects, collage, border, frame"
)

NUM_CANDIDATES = 3  # generate this many per icon; you pick the best from the contact sheet

print(f"{len(ICON_CONCEPTS)} icons queued x {NUM_CANDIDATES} candidates each: {list(ICON_CONCEPTS)}")

In [ ]:
import os

OUT_DIR = "whiteboard_icons_raw"
os.makedirs(OUT_DIR, exist_ok=True)

for name, concept in ICON_CONCEPTS.items():
    prompt = PROMPT_TEMPLATE.format(concept=concept)
    for i in range(NUM_CANDIDATES):
        generator = torch.Generator("cuda").manual_seed(i)  # fixed seeds per slot -> reproducible, comparable candidates
        image = pipe(
            prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30,
            guidance_scale=8.5,
            generator=generator,
        ).images[0]
        image.save(os.path.join(OUT_DIR, f"{name}_{i}.png"))
    print("generated", NUM_CANDIDATES, "candidates for", name)

### Pick the best candidate per icon

The contact sheet below shows every candidate (rows = icon, columns = candidate index
0..N-1). Look at it, then fill in `CHOSEN` in the next cell with the index you like best
for each icon. If *none* of an icon's candidates look right, that concept likely needs a
prompt rewrite (see `ICON_CONCEPTS` above) rather than more random seeds — Stable
Diffusion won't reliably fix a wrong concept just by re-rolling.

In [ ]:
from PIL import Image

THUMB = 200
sheet = Image.new("RGB", (THUMB * NUM_CANDIDATES, THUMB * len(ICON_CONCEPTS)), "white")
for row, name in enumerate(ICON_CONCEPTS):
    for col in range(NUM_CANDIDATES):
        thumb = Image.open(os.path.join(OUT_DIR, f"{name}_{col}.png")).convert("RGB").resize((THUMB, THUMB))
        sheet.paste(thumb, (col * THUMB, row * THUMB))
sheet.save("contact_sheet.png")
print("Rows top-to-bottom:", list(ICON_CONCEPTS))
sheet

In [ ]:
# Edit these indices after looking at the contact sheet above (0 to NUM_CANDIDATES - 1).
# Any icon left out of CHOSEN defaults to candidate 0.
CHOSEN = {
    # "piggy_bank": 1,
    # "scale": 2,
}

SELECTED = {name: CHOSEN.get(name, 0) for name in ICON_CONCEPTS}
print("Selected candidate per icon:", SELECTED)

### Remove the background (real segmentation, not a color threshold)

`rembg` runs a trained foreground/background segmentation model — it works regardless
of whether the diffusion model's background came out flat white, paper-textured, or
hatched, unlike the v1 approach (a near-white RGB threshold, which only handles the
first case).

In [ ]:
from rembg import remove

FINAL_DIR = "whiteboard_icons"
os.makedirs(FINAL_DIR, exist_ok=True)

for name, idx in SELECTED.items():
    raw = Image.open(os.path.join(OUT_DIR, f"{name}_{idx}.png"))
    cutout = remove(raw)
    cutout.save(os.path.join(FINAL_DIR, f"{name}.png"))
    print("background removed:", name, "(candidate", idx, ")")

## Part 2 — realistic hand-and-pencil

Generates a realistic hand PNG to replace the default stylized vector hand in
*every* beat's render (both vector-icon path-tracing and image/SVG reveals — see
`PHOTO_HAND_PATH` in `whiteboard_explainer.py`). Same candidate-picker + rembg
pattern as the icons above, plus one extra step: telling `make_photo_hand_cursor`
where the pencil tip actually is in your chosen image.

In [ ]:
HAND_PROMPT = (
    "a realistic human hand holding a wooden pencil, viewed from above, "
    "writing pose, pencil pointing toward the upper-left corner of the frame, "
    "photographic, isolated on a flat solid white background, no arm, no sleeve"
)
HAND_NEGATIVE_PROMPT = (
    "cartoon, drawing, sketch, illustration, painting, 3d render, textured background, "
    "paper texture, patterned background, multiple hands, extra fingers, deformed hand, "
    "blurry, low quality, watermark, text"
)
NUM_HAND_CANDIDATES = 4

HAND_OUT_DIR = "hand_raw"
os.makedirs(HAND_OUT_DIR, exist_ok=True)

for i in range(NUM_HAND_CANDIDATES):
    generator = torch.Generator("cuda").manual_seed(100 + i)
    image = pipe(
        HAND_PROMPT,
        negative_prompt=HAND_NEGATIVE_PROMPT,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    image.save(os.path.join(HAND_OUT_DIR, f"hand_{i}.png"))
print("generated", NUM_HAND_CANDIDATES, "hand candidates")

In [ ]:
hand_sheet = Image.new("RGB", (THUMB * NUM_HAND_CANDIDATES, THUMB), "white")
for col in range(NUM_HAND_CANDIDATES):
    thumb = Image.open(os.path.join(HAND_OUT_DIR, f"hand_{col}.png")).convert("RGB").resize((THUMB, THUMB))
    hand_sheet.paste(thumb, (col * THUMB, 0))
hand_sheet.save("hand_contact_sheet.png")
hand_sheet

In [ ]:
# Pick your favorite hand candidate (0 to NUM_HAND_CANDIDATES - 1) from the sheet above.
CHOSEN_HAND = 0

hand_raw = Image.open(os.path.join(HAND_OUT_DIR, f"hand_{CHOSEN_HAND}.png"))
hand_cutout = remove(hand_raw)
hand_cutout.save(os.path.join(FINAL_DIR, "hand.png"))
print("saved", os.path.join(FINAL_DIR, "hand.png"))
hand_cutout

### Calibrate the pencil-tip position

Look at the `hand.png` preview above. `whiteboard_assets.py`'s `make_photo_hand_cursor`
needs to know where the pencil tip sits, as a fraction of (half-width, half-height) from
the image's *center* — e.g. `(-0.35, 0.4)` (the default, matching the prompt's "pointing
toward the upper-left" framing) means 35% of the way from center to the left edge, 40%
of the way up from center.

If your generated hand's tip ends up somewhere clearly different, estimate the new
fraction from the preview and update `DEFAULT_PHOTO_HAND_TIP_FRACTION` in
`skills/fin-anim/scripts/scenes/whiteboard_assets.py` to match (or pass
`tip_fraction=(...)` directly to `make_photo_hand_cursor` for a one-off override).

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("whiteboard_icons", "zip", FINAL_DIR)
files.download("whiteboard_icons.zip")